Import libraries




In [1]:
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

Upload dataset

In [2]:
uploaded = files.upload()

Saving WA_Fn-UseC_-HR-Employee-Attrition.csv to WA_Fn-UseC_-HR-Employee-Attrition.csv


Load and inspect dataset

In [3]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

print("Dataset Shape:", df.shape)

display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

Dataset Shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2



Column Names:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


Check missing values

In [4]:
missing = df.isnull().sum()

print("Missing Values:")
display(missing[missing > 0])

print("Total Missing Values:", df.isnull().sum().sum())

Missing Values:


,0


Total Missing Values: 0


Handle missing values

In [5]:
df_clean = df.copy()

numeric_cols = df_clean.select_dtypes(include=np.number).columns
categorical_cols = df_clean.select_dtypes(include='object').columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Total missing values after cleaning:",
      df_clean.isnull().sum().sum())

Total missing values after cleaning: 0


Check and remove duplicates

In [6]:
duplicate_count = df_clean.duplicated().sum()

print("Duplicate records found:", duplicate_count)

df_clean = df_clean.drop_duplicates()

print("Dataset shape after removing duplicates:",
      df_clean.shape)

Duplicate records found: 0
Dataset shape after removing duplicates: (1470, 35)


Detect outliers

In [7]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns

outlier_summary = {}

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((df_clean[col] < lower) |
             (df_clean[col] > upper)).sum()

    outlier_summary[col] = count

outlier_df = pd.DataFrame(
    list(outlier_summary.items()),
    columns=['Column', 'Outlier Count']
)

display(outlier_df)

,Column,Outlier Count
0,Age,0
1,DailyRate,0
2,DistanceFromHome,0
3,Education,0
4,EmployeeCount,0
5,EmployeeNumber,0
6,EnvironmentSatisfaction,0
7,HourlyRate,0
8,JobInvolvement,0
9,JobLevel,0


Handle outliers

In [8]:
df_processed = df_clean.copy()

for col in numeric_cols:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_processed[col] = df_processed[col].clip(
        lower=lower,
        upper=upper
    )

print("Outliers handled using IQR capping.")

Outliers handled using IQR capping.


Feature engineering

In [9]:
df_processed['PromotionRatio'] = (
    df_processed['YearsSinceLastPromotion'] /
    (df_processed['TotalWorkingYears'] + 1)
)

df_processed['JobChangeRatio'] = (
    df_processed['NumCompaniesWorked'] /
    (df_processed['TotalWorkingYears'] + 1)
)

df_processed['TrainingFrequency'] = (
    df_processed['TrainingTimesLastYear'] /
    (df_processed['YearsAtCompany'] + 1)
)

df_processed['CompanyExperienceRatio'] = (
    df_processed['YearsAtCompany'] /
    (df_processed['TotalWorkingYears'] + 1)
)

df_processed['PromotionGap'] = (
    df_processed['YearsAtCompany'] -
    df_processed['YearsSinceLastPromotion']
)

print("New HR features created successfully.")

display(df_processed[
    [
        'PromotionRatio',
        'JobChangeRatio',
        'TrainingFrequency',
        'CompanyExperienceRatio',
        'PromotionGap'
    ]
].head())

New HR features created successfully.


,PromotionRatio,JobChangeRatio,TrainingFrequency,CompanyExperienceRatio,PromotionGap
0,0.000000,0.888889,0.071429,0.666667,6.0
1,0.090909,0.090909,0.272727,0.909091,9.0
2,0.000000,0.750000,3.000000,0.000000,0.0
3,0.333333,0.111111,0.333333,0.888889,5.0
4,0.285714,1.214286,1.000000,0.285714,0.0


Separate features and target

In [10]:
X = df_processed.drop('Attrition', axis=1)
y = df_processed['Attrition']

numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include='object'
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 31
Categorical features: 8


Create preprocessing pipeline

In [11]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


Train/test split and apply pipeline

In [14]:
X_all_processed = pipeline.fit_transform(X)

feature_names = pipeline.named_steps[
    'preprocessor'
].get_feature_names_out()

processed_df = pd.DataFrame(
    X_all_processed,
    columns=feature_names
)

processed_df['Attrition'] = y.reset_index(drop=True)

print("Final processed dataset shape:",
      processed_df.shape)

display(processed_df.head())

Final processed dataset shape: (1470, 61)


,numeric__Age,numeric__DailyRate,numeric__DistanceFromHome,numeric__Education,numeric__EmployeeCount,numeric__EmployeeNumber,numeric__EnvironmentSatisfaction,numeric__HourlyRate,numeric__JobInvolvement,numeric__JobLevel,...,categorical__JobRole_Research Scientist,categorical__JobRole_Sales Executive,categorical__JobRole_Sales Representative,categorical__MaritalStatus_Divorced,categorical__MaritalStatus_Married,categorical__MaritalStatus_Single,categorical__Over18_Y,categorical__OverTime_No,categorical__OverTime_Yes,Attrition
0,0.446350,0.742527,-1.010909,-0.891688,0.0,-1.701283,-0.660531,1.383138,0.379672,-0.057788,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,Yes
1,1.322365,-1.297775,-0.147150,-1.868426,0.0,-1.699621,0.254625,-0.240677,-1.026167,-0.057788,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,No
2,0.008343,1.414363,-0.887515,-0.891688,0.0,-1.696298,1.169781,1.284725,-1.026167,-0.961486,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,Yes
3,-0.429664,1.461466,-0.764121,1.061787,0.0,-1.694636,1.169781,-0.486709,0.379672,-0.961486,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,No
4,-1.086676,-0.524295,-0.887515,-1.868426,0.0,-1.691313,-1.575686,-1.274014,0.379672,-0.961486,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,No


Final verification

In [15]:
print("========== FINAL VERIFICATION ==========")

print("Rows:", processed_df.shape[0])
print("Columns:", processed_df.shape[1])

print("Missing values:",
      processed_df.isnull().sum().sum())

print("Duplicate rows:",
      processed_df.duplicated().sum())

print("\nData types:")
print(processed_df.dtypes.value_counts())

========== FINAL VERIFICATION ==========
Rows: 1470
Columns: 61
Missing values: 0
Duplicate rows: 0

Data types:
float64    60
object      1
Name: count, dtype: int64


Export clean dataset

In [16]:
output_file = 'processed_employee_attrition.csv'

processed_df.to_csv(
    output_file,
    index=False
)

print("Processed dataset saved as:",
      output_file)

Processed dataset saved as: processed_employee_attrition.csv


Download dataset

In [17]:
files.download('processed_employee_attrition.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>